<a href="https://colab.research.google.com/github/jessie0707a/Machine-Learning-Project/blob/main/Predicting_Delivery_Duration_for_Olist.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#Mount the googledrive to access files
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


Join tables

In [20]:
import pandas as pd
import os

# Base path for the datasets (assuming the CSVs are directly in this directory)
# Re-setting to the known working path based on successful previous executions.
base_path = "/content/drive/MyDrive/Machine learning/Olist order data/"

# Define file paths for each dataset
orders_path = os.path.join(base_path, "olist_orders_dataset.csv")
order_items_path = os.path.join(base_path, "olist_order_items_dataset.csv")
customers_path = os.path.join(base_path, "olist_customers_dataset.csv")
sellers_path = os.path.join(base_path, "olist_sellers_dataset.csv")
products_path = os.path.join(base_path, "olist_products_dataset.csv")
geolocation_path = os.path.join(base_path, "olist_geolocation_dataset.csv")

# Load datasets
try:
    orders_df = pd.read_csv(orders_path)
    order_items_df = pd.read_csv(order_items_path)
    customers_df = pd.read_csv(customers_path)
    sellers_df = pd.read_csv(sellers_path)
    products_df = pd.read_csv(products_path)
    geolocation_df = pd.read_csv(geolocation_path)
    print("All datasets loaded successfully.")

    # Pre-process geolocation data: aggregate by zip code to get mean lat/lng
    # This prevents an explosion of rows when merging, as each zip code can have multiple entries.
    geolocation_df_agg = geolocation_df.groupby('geolocation_zip_code_prefix').agg(
        geolocation_lat=('geolocation_lat', 'mean'),
        geolocation_lng=('geolocation_lng', 'mean')
    ).reset_index()

    # Start merging: orders and customers
    df_combined = pd.merge(orders_df, customers_df, on='customer_id', how='left')

    # Merge with order_items
    df_combined = pd.merge(df_combined, order_items_df, on='order_id', how='left')

    # Merge with products
    df_combined = pd.merge(df_combined, products_df, on='product_id', how='left')

    # Merge with sellers
    df_combined = pd.merge(df_combined, sellers_df, on='seller_id', how='left')

    # Merge with customer geolocation
    # Rename geolocation columns to distinguish from seller geolocation
    customer_geolocation_df = geolocation_df_agg.rename(columns={
        'geolocation_lat': 'customer_geolocation_lat',
        'geolocation_lng': 'customer_geolocation_lng'
    })
    df_combined = pd.merge(
        df_combined,
        customer_geolocation_df,
        left_on='customer_zip_code_prefix',
        right_on='geolocation_zip_code_prefix',
        how='left'
    )
    # Drop the redundant geolocation_zip_code_prefix column from the merged dataframe
    df_combined.drop('geolocation_zip_code_prefix', axis=1, inplace=True)

    # Merge with seller geolocation
    # Rename geolocation columns for sellers
    seller_geolocation_df = geolocation_df_agg.rename(columns={
        'geolocation_lat': 'seller_geolocation_lat',
        'geolocation_lng': 'seller_geolocation_lng'
    })
    df_combined = pd.merge(
        df_combined,
        seller_geolocation_df,
        left_on='seller_zip_code_prefix',
        right_on='geolocation_zip_code_prefix',
        how='left'
    )
    # Drop the redundant geolocation_zip_code_prefix column from the merged dataframe
    df_combined.drop('geolocation_zip_code_prefix', axis=1, inplace=True)

    print(f"Combined DataFrame created with {df_combined.shape[0]} rows and {df_combined.shape[1]} columns.")
    print("First 5 rows of the combined DataFrame:")
    print(df_combined.head())

except FileNotFoundError as e:
    print(f"Error: One of the specified files was not found. Please check the path and filename: {e}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

All datasets loaded successfully.
Combined DataFrame created with 113425 rows and 33 columns.
First 5 rows of the combined DataFrame:
                           order_id                       customer_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   
2  47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089   
3  949d5b44dbf5de918fe9c16f97b45f8a  f88197465ea7920adcdbec7375364d82   
4  ad21c59c0840e6cb83a9ceb5573f8159  8ab97904e6daea8866dbdbc4fb7aad2c   

  order_status order_purchase_timestamp    order_approved_at  \
0    delivered      2017-10-02 10:56:33  2017-10-02 11:07:15   
1    delivered      2018-07-24 20:41:37  2018-07-26 03:24:27   
2    delivered      2018-08-08 08:38:49  2018-08-08 08:55:23   
3    delivered      2017-11-18 19:28:06  2017-11-18 19:45:59   
4    delivered      2018-02-13 21:18:39  2018-02-13 22:20:29   

  order_delivered_carrier_date order_deliv

In [13]:
display(df_combined.head())

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,product_length_cm,product_height_cm,product_width_cm,seller_zip_code_prefix,seller_city,seller_state,customer_geolocation_lat,customer_geolocation_lng,seller_geolocation_lat,seller_geolocation_lng
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,7c396fd4830fd04220f754e42b4e5bff,3149,...,19.0,8.0,13.0,9350.0,maua,SP,-23.576983,-46.587161,-23.680729,-46.444238
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,af07308b275d755c9edb36a90c618231,47813,...,19.0,13.0,19.0,31570.0,belo horizonte,SP,-12.177924,-44.660711,-19.807681,-43.980427
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,3a653a41f6f9fc3d2a113cf8398680e8,75265,...,24.0,19.0,21.0,14840.0,guariba,SP,-16.745150,-48.514783,-21.363502,-48.229601
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,7c142cf63193a1473d2e66489a9ae977,59296,...,30.0,10.0,20.0,31842.0,belo horizonte,MG,-5.774190,-35.271143,-19.837682,-43.924053
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,72632f0f9dd73dfee390c9b22eb56dd6,9195,...,51.0,15.0,15.0,8752.0,mogi das cruzes,SP,-23.676370,-46.514627,-23.543395,-46.262086


In [14]:
print(df_combined.columns.tolist())

['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm', 'seller_zip_code_prefix', 'seller_city', 'seller_state', 'customer_geolocation_lat', 'customer_geolocation_lng', 'seller_geolocation_lat', 'seller_geolocation_lng']


Here's the list of columns that will be kept in the `df_combined` based on your instructions. I will also load and merge the `olist_order_payments_dataset`.

### Payments Data Processing

For the `olist_order_payments_dataset`, I'll aggregate by `order_id` to prevent duplicating rows in the main DataFrame. I will take the `first` `payment_type` and sum the `payment_installments` for each order, as multiple payment entries for an order might contribute to the total installments.

In [21]:
# Define path for payments dataset
payments_path = os.path.join(base_path, "olist_order_payments_dataset.csv")

# Load payments dataset
try:
    payments_df = pd.read_csv(payments_path)
    print("Payments dataset loaded successfully.")

    # Select and aggregate payments data
    # Aggregate by order_id: take the first payment_type and sum payment_installments
    payments_agg_df = payments_df.groupby('order_id').agg(
        payment_type=('payment_type', lambda x: x.mode()[0] if not x.mode().empty else None), # Use mode for categorical, handling empty mode
        payment_installments=('payment_installments', 'sum')
    ).reset_index()

    # Merge aggregated payments with df_combined
    df_combined = pd.merge(df_combined, payments_agg_df, on='order_id', how='left')
    print("Payments data merged into df_combined.")

except FileNotFoundError as e:
    print(f"Error: The payments file was not found. Please check the path and filename: {e}")
except Exception as e:
    print(f"An unexpected error occurred during payments processing: {e}")

Payments dataset loaded successfully.
Payments data merged into df_combined.


filter the `df_combined` to keep only the columns specified

In [22]:
columns_to_keep = [
    'order_id',
    'customer_id',
    'order_purchase_timestamp',
    'order_delivered_customer_date', # Kept for target variable calculation
    'order_estimated_delivery_date',
    'product_id',
    'seller_id',
    'shipping_limit_date',
    'price',
    'freight_value',
    'customer_zip_code_prefix',
    'customer_state',
    'seller_zip_code_prefix',
    'seller_state',
    'customer_geolocation_lat',
    'customer_geolocation_lng',
    'seller_geolocation_lat',
    'seller_geolocation_lng',
    'product_category_name',
    'product_weight_g',
    'product_length_cm',
    'product_height_cm',
    'product_width_cm',
    'payment_type',
    'payment_installments'
]

# Filter df_combined to keep only the specified columns
df_processed = df_combined[columns_to_keep].copy()

# Convert date columns to datetime objects for easier manipulation
date_cols = ['order_purchase_timestamp', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'shipping_limit_date']
for col in date_cols:
    if col in df_processed.columns:
        df_processed[col] = pd.to_datetime(df_processed[col], errors='coerce')

print(f"Processed DataFrame created with {df_processed.shape[0]} rows and {df_processed.shape[1]} columns.")
print("First 5 rows of the processed DataFrame:")
display(df_processed.head())

print("Column data types of the processed DataFrame:")
display(df_processed.info())

Processed DataFrame created with 113425 rows and 25 columns.
First 5 rows of the processed DataFrame:


,order_id,customer_id,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,product_id,seller_id,shipping_limit_date,price,freight_value,...,customer_geolocation_lng,seller_geolocation_lat,seller_geolocation_lng,product_category_name,product_weight_g,product_length_cm,product_height_cm,product_width_cm,payment_type,payment_installments
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,2017-10-02 10:56:33,2017-10-10 21:25:13,2017-10-18,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72,...,-46.587161,-23.680729,-46.444238,utilidades_domesticas,500.0,19.0,8.0,13.0,voucher,3.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,2018-07-24 20:41:37,2018-08-07 15:27:45,2018-08-13,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,2018-07-30 03:24:27,118.70,22.76,...,-44.660711,-19.807681,-43.980427,perfumaria,400.0,19.0,13.0,19.0,boleto,1.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,2018-08-08 08:38:49,2018-08-17 18:06:29,2018-09-04,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,2018-08-13 08:55:23,159.90,19.22,...,-48.514783,-21.363502,-48.229601,automotivo,420.0,24.0,19.0,21.0,credit_card,3.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,2017-11-18 19:28:06,2017-12-02 00:28:42,2017-12-15,d0b61bfb1de832b15ba9d266ca96e5b0,66922902710d126a0e7d26b0e3805106,2017-11-23 19:45:59,45.00,27.20,...,-35.271143,-19.837682,-43.924053,pet_shop,450.0,30.0,10.0,20.0,credit_card,1.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,2018-02-13 21:18:39,2018-02-16 18:17:02,2018-02-26,65266b2da20d04dbe00c5c2d3bb7859e,2c9e548be18521d1c43cde1c582c6de8,2018-02-19 20:31:37,19.90,8.72,...,-46.514627,-23.543395,-46.262086,papelaria,250.0,51.0,15.0,15.0,credit_card,1.0


Column data types of the processed DataFrame:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 113425 entries, 0 to 113424
Data columns (total 25 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   order_id                       113425 non-null  object        
 1   customer_id                    113425 non-null  object        
 2   order_purchase_timestamp       113425 non-null  datetime64[ns]
 3   order_delivered_customer_date  110196 non-null  datetime64[ns]
 4   order_estimated_delivery_date  113425 non-null  datetime64[ns]
 5   product_id                     112650 non-null  object        
 6   seller_id                      112650 non-null  object        
 7   shipping_limit_date            112650 non-null  datetime64[ns]
 8   price                          112650 non-null  float64       
 9   freight_value                  112650 non-null  float64       
 10  customer_zip_code_pref

None